# UCSD-PTGBM - exploratory data analysis

[TCIA collection UCSD-PTGBM](https://www.cancerimagingarchive.net/collection/ucsd-ptgbm/):
178 patients, 243 **post-treatment** timepoints, with expert labels
separating residual/recurrent tumour from treatment-related change - and an
explicit record of how each negative judgement was made.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from gbmplast.adapters import get_adapter
from gbmplast.config import load_config
from gbmplast.report import dictionary as D
from gbmplast.report.theme import apply_theme, series_color
from gbmplast.validate import validate_bundle

apply_theme()
config = load_config()
adapter = get_adapter("ucsd_ptgbm", config)
bundle = adapter.build()
bundle.summary()

,table,n_rows,n_cols
0,patients,178,23
1,visits,243,27
2,imaging,1701,25
3,treatments,347,13
4,symptoms,0,15
5,molecular_samples,0,15
6,outcomes,480,15


## The cohort's own data dictionary

Read it rather than inferring column meanings. It states, for example, that
`Overall survival` is "days between initial surgery and death".

In [2]:
adapter.data_dictionary_official()

,column,description
0,ID,TCIA ID
1,Brats Subject ID,ID used in the BraTS 2024 challenge
2,Magnetic Field Strength,Magnetic field strength of the scanner
3,Manufacturer,Manufacturer of the scanner
4,Manufacturer's Model Name,Model Name of the scanner
...,...,...
67,Date of 1st chemo start,NaN
68,Avastin® (bevacizumab),NaN
69,Date of last Avastin treatment,NaN
70,Other treatment,NaN


## Time anchoring

Every released offset is **days from that scan's acquisition** and may be
negative. The adapter re-anchors to initial surgery so that this cohort
shares one convention with the others.

In [3]:
visits = bundle.visits
print("timepoints per patient:",
      visits.groupby("patient_id").size().value_counts().to_dict())
print(f"\nvisits with a known offset from surgery: "
      f"{int(visits.relative_time_days.notna().sum())} of {len(visits)}")
visits[["local_visit_id", "timepoint_index", "relative_time_days",
        "days_acq_to_last_surgery", "days_acq_to_last_radiation"]].head(8)

timepoints per patient: {1: 129, 2: 35, 3: 12, 4: 2}

visits with a known offset from surgery: 198 of 243


,local_visit_id,timepoint_index,relative_time_days,days_acq_to_last_surgery,days_acq_to_last_radiation
0,UCSD-PTGBM-0001_01,1,647,-336.0,-914.0
1,UCSD-PTGBM-0002_01,1,196,NaN,-97.0
2,UCSD-PTGBM-0003_01,1,117,NaN,-30.0
3,UCSD-PTGBM-0004_01,1,25,NaN,NaN
4,UCSD-PTGBM-0004_02,2,803,-144.0,-18.0
5,UCSD-PTGBM-0005_01,1,29,NaN,NaN
6,UCSD-PTGBM-0005_02,2,341,NaN,-252.0
7,UCSD-PTGBM-0006_01,1,105,NaN,-28.0


## The outcome, and how it was adjudicated

This is the cohort's distinguishing feature. 192 timepoints are
tumour-positive and 51 are treatment-related change - and the negatives carry
the **basis** of the judgement.

In [4]:
print(visits.tumor_status.value_counts().to_string())
print()
pd.crosstab(visits.tumor_status_detail, visits.tumor_status_ascertainment)

tumor_status
tumor_positive              192
treatment_related_change     51



tumor_status_ascertainment,clinical_consensus,histopathology,radiologic_followup,unknown
tumor_status_detail,,,,
nonspecific_treatment_change,0,0,29,0
pseudoprogression,9,6,0,0
radiation_necrosis,0,0,0,7
residual_or_recurrent_tumor,0,0,0,192


A limitation worth stating plainly: the collection abstract says 45 of the
192 positives are pathology-proven, but **no released column flags which
ones**. So a histopathology-restricted comparison is possible on the negative
class only, and this pipeline refuses to construct one for the positives.

In [5]:
outcomes = bundle.outcomes
tumour = outcomes[outcomes.outcome_name == "tumor_vs_treatment_change"]
tumour.groupby(["outcome_label", "ascertainment"]).size()

outcome_label             ascertainment      
treatment_related_change  clinical_consensus       9
                          histopathology           6
                          radiologic_followup     29
                          unknown                  7
tumor_positive            unknown                192
dtype: int64

## Treatments

Unusually for a public imaging cohort, treatment episodes are released with
relative dates.

In [6]:
treatments = bundle.treatments
print(treatments.treatment_type.value_counts().to_string())
print(f"\nepisodes with a known start day: "
      f"{int(treatments.start_relative_day.notna().sum())} of {len(treatments)}")
treatments.head(8)[["local_patient_id" if "local_patient_id" in treatments
                    else "patient_id", "treatment_type", "treatment_name",
                    "start_relative_day", "end_relative_day",
                    "extent_of_resection"]]

treatment_type
surgery         147
chemotherapy     94
radiotherapy     80
other            26

episodes with a known start day: 265 of 347


,patient_id,treatment_type,treatment_name,start_relative_day,end_relative_day,extent_of_resection
0,UCSD-PTGBM::UCSD-PTGBM-0001,surgery,resection,0,NaN,GTR
1,UCSD-PTGBM::UCSD-PTGBM-0001,radiotherapy,radiation,56,-267,NaN
2,UCSD-PTGBM::UCSD-PTGBM-0001,chemotherapy,Temodar,56,NaN,NaN
3,UCSD-PTGBM::UCSD-PTGBM-0001,other,other (unspecified),<NA>,NaN,NaN
4,UCSD-PTGBM::UCSD-PTGBM-0002,surgery,resection,0,NaN,GTR
5,UCSD-PTGBM::UCSD-PTGBM-0002,radiotherapy,radiation,56,99,NaN
6,UCSD-PTGBM::UCSD-PTGBM-0002,chemotherapy,Temodar,64,NaN,NaN
7,UCSD-PTGBM::UCSD-PTGBM-0003,surgery,resection,0,NaN,GTR


## Molecular markers

In [7]:
patients = bundle.patients
for col in ("mgmt_status", "idh_status", "codeletion_1p19q", "atrx_status"):
    print(f"{col}: {patients[col].value_counts(dropna=False).to_dict()}")

mgmt_status: {<NA>: 72, 'methylated': 53, 'unmethylated': 53}
idh_status: {'wildtype': 111, <NA>: 55, 'mutant': 12}
codeletion_1p19q: {None: 98, 'Intact': 74, 'Codeleted': 6}
atrx_status: {None: 94, 'Intact': 70, 'Loss': 14}


## Survival

OS is populated for a subset; PFS has a time but **no progression-event
indicator**, so it cannot be used for time-to-event modelling.

In [8]:
survival = outcomes[outcomes.outcome_type == "time_to_event"]
survival.groupby("outcome_name").agg(
    n=("outcome_value", "size"),
    n_with_time=("outcome_value", lambda s: int(s.notna().sum())),
    n_events=("event_indicator", lambda s: int((s == True).sum())),
    n_censoring_unknown=("event_indicator", lambda s: int(s.isna().sum())))

,n,n_with_time,n_events,n_censoring_unknown
outcome_name,,,,
overall_survival,178,148,148,0
progression_free_survival,59,59,0,59


One released PFS value is negative (-8 days), which is an error in the source
table. It is reported rather than silently dropped or clipped.

In [9]:
negative = survival[pd.to_numeric(survival.outcome_value,
                                  errors="coerce") < 0]
negative[["patient_id", "outcome_name", "outcome_value", "event_indicator"]]

,patient_id,outcome_name,outcome_value,event_indicator
199,UCSD-PTGBM::UCSD-PTGBM-0155,progression_free_survival,-8.0,<NA>


## Imaging

The protocol is documented, but the clinical tables carry no per-scan
availability flags, so `available` is **NA (unknown)** rather than True.
Imaging itself needs IBM Aspera and is not retrievable headlessly.

In [10]:
D.modality_completeness(bundle)

,cohort,modality,n_rows,n_available,n_unavailable,n_unknown,pct_available,n_visits_with_modality,n_patients_with_modality
0,ucsd_ptgbm,ASL,243,0,0,243,0.0,0,0
1,ucsd_ptgbm,DSC,243,0,0,243,0.0,0,0
2,ucsd_ptgbm,FLAIR,243,0,0,243,0.0,0,0
3,ucsd_ptgbm,RSI,243,0,0,243,0.0,0,0
4,ucsd_ptgbm,SEG,243,0,0,243,0.0,0,0
5,ucsd_ptgbm,T1,243,0,0,243,0.0,0,0
6,ucsd_ptgbm,T1CE,243,0,0,243,0.0,0,0


In [11]:
report = validate_bundle(bundle)
report.to_frame()[["check", "table", "severity", "message"]]

,check,table,severity,message
0,primary_key_unique,patients,info,primary key ['patient_id'] unique over 178 rows
1,primary_key_unique,visits,info,primary key ['visit_id'] unique over 243 rows
2,primary_key_unique,imaging,info,primary key ['imaging_id'] unique over 1701 rows
3,primary_key_unique,treatments,info,primary key ['treatment_id'] unique over 347 rows
4,primary_key_unique,outcomes,info,primary key ['outcome_id'] unique over 480 rows
5,visit_time_known,visits,warning,45 visits have no relative_time_days; they are...
6,longitudinal_ordering,visits,info,visit_index consistent with time for all 178 p...
7,survival_negative_time,outcomes,warning,1 time-to-event rows have negative times in th...
8,modality_availability_unknown,imaging,warning,1701 imaging rows have unknown availability (m...
